# Model Training - ChatKasir

- Nama: Achmad Rif'an
- Bagian: AI-1 (Model Architect)

## 1. Import Library

In [1]:
import os
import time
import json
import contextlib
import numpy as np
import tensorflow as tf

from google.colab import drive
drive.mount('/content/drive')

from tensorflow.keras.layers import Input, Embedding, Dense, Dropout, LayerNormalization, MultiHeadAttention, GlobalAveragePooling1D
from tensorflow.keras.layers import Bidirectional, LSTM
from tensorflow.keras.models import Model

Mounted at /content/drive


## 2. Memuat Konfigurasi

Kita mengambil file model_config.json untuk mengetahui parameter model seperti vocab_size dan max_length. Ini penting agar arsitektur yang kita bangun di notebook ini sama persis dengan data yang sudah disiapkan.

In [2]:
# Memuat konfigurasi arsitektur dari file JSON
# config_path = "..\\assets\\data\\model_config.json"   # path lokal (Windows)
config_path = "/content/drive/MyDrive/ChatKasir/assets/data/model_config.json"  # path di Google Drive Colab

# Membuka file konfigurasi dan membaca isinya sebagai dictionary Python
with open(config_path, "r") as f:
    config = json.load(f)

# Mengambil variabel penting dari konfigurasi
VOCAB_SIZE = config['vocab_size']       # ukuran kosakata untuk embedding
MAX_LENGTH = config['max_length']       # panjang maksimum sequence input
NUM_TAGS = config['num_product_tags']   # jumlah kategori/label produk

print(f"Konfigurasi dimuat: Vocab={VOCAB_SIZE}, Max Length={MAX_LENGTH}, Num Tags={NUM_TAGS}")

Konfigurasi dimuat: Vocab=5000, Max Length=64, Num Tags=3


## 3. Data Loading

1. Memuat Dataset: Kita memuat file dataset_chatkasir.npz yang berisi array NumPy untuk data Training, Validation, dan Testing.

2. Membuat tf.data.Dataset: Kita mengubah array tersebut menjadi objek Dataset TensorFlow. Ini adalah cara paling efisien untuk melatih model karena mendukung fitur shuffling (mengacak data) dan batching (mengambil data sedikit demi sedikit) agar tidak membebani memori RAM.

In [3]:
# Memuat dataset yang sudah dibagi (Train, Val, Test)
# data_path = "..\\assets\\data\\dataset_chatkasir.npz"   # path lokal (Windows)
data_path = "/content/drive/MyDrive/ChatKasir/assets/data/dataset_chatkasir.npz"  # path di Google Drive Colab
data = np.load(data_path)  # memuat file .npz berisi dataset

# Ekstrak data Training
X_train = data['X_train']             # input teks untuk training
Y_prod_train = data['Y_prod_train']   # label produk untuk training
Y_qty_train = data['Y_qty_train']     # label jumlah untuk training

# Normalisasi harga (dibagi 1000)
# Jika harga bukan -1, bagi dengan 1000. Jika -1, biarkan tetap -1 (menandakan tidak ada harga)
Y_price_train_raw = data['Y_price_train']
Y_price_train = np.where(Y_price_train_raw != -1.0, Y_price_train_raw / 1000.0, -1.0)

# Ekstrak data Validation
X_val = data['X_val']             # input teks untuk validasi
Y_prod_val = data['Y_prod_val']   # label produk untuk validasi
Y_qty_val = data['Y_qty_val']     # label jumlah untuk validasi

# Normalisasi juga untuk data Validation
Y_price_val_raw = data['Y_price_val']
Y_price_val = np.where(Y_price_val_raw != -1.0, Y_price_val_raw / 1000.0, -1.0)

print(f"Dataset dimuat: Training={len(X_train)} baris, Validation={len(X_val)} baris")


Dataset dimuat: Training=80400 baris, Validation=10050 baris


In [4]:
# Mengonversi ke tf.data.Dataset untuk efisiensi training
BATCH_SIZE = 32 # Jumlah data yang diproses sekali epoch

def create_tf_dataset(X, y_prod, y_qty, y_price, is_training=True):
    # Gabungkan Input (X) dengan 3 Target (Y)
    ds = tf.data.Dataset.from_tensor_slices((X, (y_prod, y_qty, y_price)))

    if is_training:
        ds = ds.shuffle(10000) # Acak data agar model tidak menghafal urutan

    # Ambil data per batch dan siapkan batch berikutnya di latar belakang (prefetch)
    # AUTOTUNE = otomatis mengatur penggunaan CPU/GPU
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

# Buat objek dataset untuk Training dan Validation
train_ds = create_tf_dataset(X_train, Y_prod_train, Y_qty_train, Y_price_train)
val_ds = create_tf_dataset(X_val, Y_prod_val, Y_qty_val, Y_price_val, is_training=False)

print("Objek tf.data.Dataset berhasil dibuat")

Objek tf.data.Dataset berhasil dibuat


## 4. Re-build Model

Meskipun kita sudah merancang model di notebook sebelumnya, kita perlu mendefinisikan ulang strukturnya di notebook ini agar objek model tersebut tercipta kembali di memori sebelum dilatih.

Ada beberapa hal penting yang kita lakukan di sini:

1. Mendefinisikan TransformerEncoder: Kita menyertakan kembali kelas kustom ini, lengkap dengan dukungan masking agar model tidak "bingung" melihat token padding.

2. Mendefinisikan model_transformer: Fungsi ini membangun arsitektur Multi-Task kita yang terdiri dari satu tulang punggung (backbone) Transformer dan tiga cabang prediksi (Produk, Jumlah, Harga).

3. Instansiasi Model: Kita memanggil fungsi tersebut menggunakan variabel VOCAB_SIZE, MAX_LENGTH, dan NUM_TAGS yang sudah kita muat dari file konfigurasi di Tahap 1.

In [5]:
# 1. CUSTOM LAYER: TRANSFORMER ENCODER
# Membaca seluruh kalimat sekaligus dan mencari tahu hubungan antar kata (konteks)
class TransformerEncoder(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1, **kwargs):
        super(TransformerEncoder, self).__init__(**kwargs)
        # Layer tidak akan menghitung token kosong (padding)
        self.supports_masking = True

        # Multi-Head Attention: Fitur utama Transformer. Membantu model fokus
        # pada kata-kata penting. Misal saat melihat kata "es", ia tahu harus
        # memperhatikan kata "teh" di sebelahnya.
        self.att = MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)

        # Feed-Forward Network: untuk memproses lebih lanjut informasi yang didapat dari Attention.
        self.ffn = tf.keras.Sequential([Dense(ff_dim, activation="relu"), Dense(embed_dim)])

        # Layer Normalization & Dropout
        self.layernorm1 = LayerNormalization(epsilon=1e-6)
        self.layernorm2 = LayerNormalization(epsilon=1e-6)
        self.dropout1 = Dropout(rate)
        self.dropout2 = Dropout(rate)

    def call(self, inputs, training=False, mask=None):
        # MASKING: Jika kalimat aslinya pendek, sisa ruangnya diisi angka 0 (PAD).
        # Bagian ini akan mengabaikan angka 0 tersebut.
        padding_mask = tf.cast(mask[:, tf.newaxis, :], dtype=tf.int32) if mask is not None else None

        # Tahap 1: Evaluasi hubungan antar kata (Attention)
        attn_output = self.att(inputs, inputs, attention_mask=padding_mask)
        attn_output = self.dropout1(attn_output, training=training)
        # Skip-connection (Residual): Menambahkan input asli ke output agar memori kata tidak hilang
        out1 = self.layernorm1(inputs + attn_output)

        # Tahap 2: Pemrosesan lanjutan (Feed Forward)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

# 2. FUNGSI ARSITEKTUR MULTI-TASK LEARNING
# Menggabungkan Transformer dengan 3 cabang: Tebak Produk, Jumlah, Harga
def model_transformer(vocab_size, max_length, num_product_tags):
    # Hyperparameter
    embed_dim = 128
    num_heads = 4
    ff_dim = 256

    # --- INPUT LAYER ---
    # Menerima daftar ID kata dengan panjang tetap
    inputs = Input(shape=(max_length,), name="input_ids")

    # --- SHARED BACKBONE ---
    # Embedding: Mengubah ID kata (angka) menjadi vektor makna (ruang vektor).
    # mask_zero=True: Otomatis memberi tahu seluruh layer di bawahnya untuk mengabaikan angka 0.
    x = Embedding(input_dim=vocab_size, output_dim=embed_dim, mask_zero=True)(inputs)

    # Memasukkan vektor kata ke dalam 2 blok Transformer agar model makin paham konteks
    x = TransformerEncoder(embed_dim, num_heads, ff_dim)(x)
    x = TransformerEncoder(embed_dim, num_heads, ff_dim)(x)

    # GlobalAveragePooling: Merangkum makna dari 64 kata menjadi 1 kesimpulan tunggal.
    # dipakai untuk cabang regresi (menebak angka tunggal seperti jumlah/harga).
    x_pooled = GlobalAveragePooling1D()(x)

    # --- CABANG 1: MENEBAK NAMA PRODUK (NER) ---
    # Memakai Bidirectional LSTM (Membaca urutan kata dari kiri ke kanan dan kanan ke kiri)
    # return_sequences=True memastikan mengeluarkan tebakan untuk tiap kata,
    # bukan cuma 1 tebakan di akhir kalimat.
    branch_product = Bidirectional(LSTM(64, return_sequences=True))(x)
    branch_product = Dense(64, activation='relu')(branch_product)
    # Output berupa probabilitas (Softmax) karena tugasnya mengklasifikasi kategori (O, B-PROD, I-PROD)
    output_product = Dense(num_product_tags, activation='softmax', name="product_tags")(branch_product)

    # --- CABANG 2: MENEBAK JUMLAH (QUANTITY) ---
    # Mengambil kesimpulan kalimat (x_pooled) lalu menebak sebuah angka pasti (Regresi).
    # Aktivasi ReLU dipakai karena jumlah pesanan tidak mungkin minus.
    branch_quantity = Dense(32, activation='relu')(x_pooled)
    output_quantity = Dense(1, activation='relu', name="quantity")(branch_quantity)

    # --- CABANG 3: MENEBAK HARGA SATUAN (PRICE) ---
    # Sama seperti cabang Quantity, menebak nilai angka kontinu (Regresi).
    branch_price = Dense(32, activation='relu')(x_pooled)
    output_price = Dense(1, activation='relu', name="price")(branch_price)

    # Satukan semuanya ke dalam satu objek Model Keras
    return Model(inputs=inputs, outputs=[output_product, output_quantity, output_price])

# 3. INSTANSIASI & SUMMARY
model = model_transformer(
    vocab_size=VOCAB_SIZE,
    max_length=MAX_LENGTH,
    num_product_tags=NUM_TAGS
)

# Tampilkan ringkasan arsitektur
model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_ids           │ (None, 64)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 64, 128)   │    640,000 │ input_ids[0][0]   │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 64)        │          0 │ input_ids[0][0]   │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_encoder │ (None, 64, 128)   │    330,240 │ embedding[0][0],  │
│ (TransformerEncode… │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_encode… │ (None, 64, 128)   │    330,240 │ transformer_enco… │
│ (TransformerEncode… │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional       │ (None, 64, 128)   │     98,816 │ transformer_enco… │
│ (Bidirectional)     │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 128)       │          0 │ transformer_enco… │
│ (GlobalAveragePool… │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 64, 64)    │      8,256 │ bidirectional[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 32)        │      4,128 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_6 (Dense)     │ (None, 32)        │      4,128 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ product_tags        │ (None, 64, 3)     │        195 │ dense_4[0][0]     │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ quantity (Dense)    │ (None, 1)         │         33 │ dense_5[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ price (Dense)       │ (None, 1)         │         33 │ dense_6[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,416,069 (5.40 MB)

 Trainable params: 1,416,069 (5.40 MB)

 Non-trainable params: 0 (0.00 B)

## 5. Loss Function & Optimizer
1. Loss Function: Ini adalah rumus matematika untuk menghitung seberapa jauh tebakan model dari kenyataan.

   - Cabang Produk: Menggunakan Sparse Categorical Crossentropy karena tugasnya adalah klasifikasi kategori (O, B-PROD, I-PROD).

   - Cabang Quantity: Menggunakan Mean Absolute Error (MAE) karena tugasnya menebak angka kontinu.

   - Cabang Harga (MaskedPriceLoss): Ini yang paling spesial. Kita harus membuat fungsi kustom agar model mengabaikan data yang nilai harganya -1 (saat harga tidak disebutkan di chat). Jika tidak di-masking, model akan belajar menebak angka -1, padahal itu hanya penanda data kosong.

2. Optimizer: Ini adalah algoritma yang bertugas memperbaiki bobot model berdasarkan nilai loss. Kita menggunakan Adam, yang merupakan standar industri karena kecepatannya dalam belajar.

3. Dynamic Weighting Variables: Kita menyiapkan variabel pembobot awal agar nantinya model bisa menyeimbangkan fokus belajarnya antara Produk, Jumlah, dan Harga secara otomatis.

In [6]:
# 1. CUSTOM LOSS: MASKED PRICE LOSS
# Tidak semua pesan pembeli menyebutkan harga (jika tidak ada, nilainya -1).
# Jika harganya -1, model jangan dihukum meskipun tebakannya meleset.
class MaskedPriceLoss(tf.keras.losses.Loss):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        # Mean Absolute Error (MAE) mentah (belum dirata-rata).
        # reduction='none' = menghitung error per baris data,
        # tidak langsung ditotal, agar nanti bisa dipilah-pilih.
        self.mae = tf.keras.losses.MeanAbsoluteError(reduction='none')

    def call(self, y_true, y_pred):
        # Buat masker (filter):
        # Jika harga asli (y_true) bukan -1, beri nilai 1.0 (Valid / Hitung errornya)
        # Jika harga asli adalah -1, beri nilai 0.0 (Abaikan / Jangan hitung)
        mask = tf.cast(tf.not_equal(y_true, -1.0), tf.float32)

        # Hitung MAE mentah (seberapa jauh tebakan model dari kenyataan)
        loss = self.mae(y_true, y_pred)

        # Kalikan error dengan masker.
        # Jika maskernya 0 (karena harganya -1), maka errornya otomatis jadi 0 (diabaikan).
        masked_loss = loss * mask

        # Kembalikan rata-rata error hanya dari data yang valid (maskernya 1).
        # Ditambah 1e-7 (angka sangat kecil) di pembagian agar tidak terjadi error pembagian dengan nol.
        return tf.reduce_sum(masked_loss) / (tf.reduce_sum(mask) + 1e-7)

# 2. INISIALISASI LOSS FUNCTION
# SparseCategoricalCrossentropy dipakai karena label produk berupa angka ID (0, 1, 2 dst).
# reduction='none' agar bisa menghapus hukuman untuk token padding angka 0.
loss_fn_product = tf.keras.losses.SparseCategoricalCrossentropy(reduction='none')

# Untuk cabang kuantitas, pakai MAE standar (karena jumlah pesanan selalu ada nilainya)
loss_fn_quantity = tf.keras.losses.MeanAbsoluteError()

# Untuk cabang harga, pakai loss kustom
loss_fn_price = MaskedPriceLoss()

# 3. OPTIMIZER & LEARNING RATE SCHEDULER
# ExponentialDecay: Sistem untuk memperlambat langkah belajar AI.
# Di awal AI butuh lari cepat agar banyak belajar, tapi saat sudah mau
# mencapai target (MAE 0.02), harus melangkah pelan-pelan agar tidak kebablasan.
lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
    initial_learning_rate=0.001, # Kecepatan awal
    decay_steps=5000,            # Turunkan kecepatan setiap model melewati 5000 langkah (~2 epoch)
    decay_rate=0.9,              # Kurangi kecepatannya sebesar 10% dari kecepatan sebelumnya
    staircase=True               # Penurunannya tegas
)

# Adam adalah algoritma dalam mencari jalan terpendek
# untuk menurunkan loss tambahkan learning rate scheduler tadi ke dalam Adam.
optimizer = tf.keras.optimizers.Adam(learning_rate=lr_schedule)

# 4. VARIABEL DYNAMIC LOSS WEIGHTING
# tf.Variable memungkinkan nilainya diubah di tengah jalan (saat proses training berlangsung).
# trainable=False mencegah Adam mengganti nilainya sendiri.
w_product = tf.Variable(1.0, trainable=False, name="w_prod")
w_quantity = tf.Variable(1.0, trainable=False, name="w_qty")
w_price = tf.Variable(1.0, trainable=False, name="w_price")

## 6. Custom Training Loop (tf.GradientTape)

Membuat 2 fungsi utama:

1. train_step: Ini adalah rutinitas belajar model. Di sini, kita menggunakan tf.GradientTape sebagai "perekam". Saat model membuat tebakan (Forward Pass), tape merekamnya. Lalu kita hitung kesalahannya (Loss), dan kita putar balik rekaman tersebut (Backpropagation) untuk memperbaiki saraf-saraf model menggunakan Optimizer. Kita juga menerapkan Dynamic Loss Weighting dengan mengalikan loss dengan bobot w_product, w_quantity, dan w_price.

2. val_step: Ini adalah rutinitas ujian/tryout. Tidak ada tape perekam dan tidak ada perbaikan saraf. Model hanya murni menebak, lalu kita hitung seberapa meleset tebakannya.

In [7]:
# 1. INISIALISASI METRICS
# dalam menebak nama produk, pakai SparseCategoricalAccuracy karena
# label aslinya berbentuk angka/ID (0 untuk 'O', 1 untuk 'B-PROD', dst).
train_acc_metric = tf.keras.metrics.SparseCategoricalAccuracy()
val_acc_metric = tf.keras.metrics.SparseCategoricalAccuracy()

# 2. FUNGSI TRAINING
@tf.function
def train_step(x_batch, y_prod, y_qty, y_price):

    # tf.GradientTape adalah "Kamera Perekam". Saat berada di dalam blok `with` ini,
    # tape akan merekam semua perhitungan matematika yang dilakukan model.
    # Rekaman ini nanti dipakai untuk melacak "dari mana asal kesalahan tebakan".
    with tf.GradientTape() as tape:

        # Forward Pass: Model disuruh menebak
        # training=True berarti fitur seperti Dropout sedang aktif
        pred_prod, pred_qty, pred_price = model(x_batch, training=True)

        # --- MASKING & CLASS WEIGHTING (Sistem Hukuman untuk Produk) ---

        # 1. Masker Padding: Beri nilai 1 jika itu kata asli, 0 jika itu padding [PAD].
        # Agar model tidak membuang waktu menghafal urutan kosong.
        mask_padding = tf.cast(tf.not_equal(x_batch, 0), tf.float32)

        # 2. Masker Kelas (Class Weighting):
        # Karena kata Produk itu sedikit, dan kata biasa ('O') itu banyak,
        # model akan dihukum 10x lipat (10.0) jika gagal menebak Produk,
        # tapi hanya dihukum biasa (1.0) jika salah menebak kata 'O'.
        mask_class = tf.where(tf.equal(y_prod, 0), 1.0, 10.0)

        # 3. Gabungkan kedua masker di atas.
        final_mask = mask_padding * mask_class

        # Hitung kesalahan (Loss) tebakan produk, lalu kalikan dengan masker.
        loss_prod_raw = loss_fn_product(y_prod, pred_prod)
        loss_prod = tf.reduce_sum(loss_prod_raw * final_mask) / (tf.reduce_sum(final_mask) + 1e-7)

        # Hitung kesalahan tebakan Jumlah dan Harga.
        loss_qty = loss_fn_quantity(y_qty, pred_qty)
        loss_price = loss_fn_price(y_price, pred_price)

        # --- DYNAMIC LOSS WEIGHTING ---
        # Kalikan error dengan w_product, w_qty, w_price.
        # Jika w_product nilainya besar, maka total_loss akan didominasi oleh kesalahan produk.
        weighted_loss_prod = w_product * loss_prod
        weighted_loss_qty = w_quantity * loss_qty
        weighted_loss_price = w_price * loss_price

        # Jumlahkan semua kesalahan menjadi satu Total Loss
        total_loss = weighted_loss_prod + weighted_loss_qty + weighted_loss_price

    # --- BACKPROPAGATION ---
    # Tape memutar balik rekaman, mencari tahu saraf mana yang menyebabkan Total Loss besar.
    gradients = tape.gradient(total_loss, model.trainable_weights)

    # Optimizer (Adam) mengambil hasil rekaman itu, lalu sedikit mengubah bobot/saraf
    # model agar di tebakan berikutnya kesalahannya mengecil.
    optimizer.apply_gradients(zip(gradients, model.trainable_weights))

    # Update metric (hanya pakai mask_padding di sini,
    # agar nilai akurasinya murni 1 kata 1 poin, tidak dikali 10).
    train_acc_metric.update_state(y_prod, pred_prod, sample_weight=mask_padding)

    return total_loss, loss_prod, loss_qty, loss_price

# 3. FUNGSI VALIDATION
@tf.function
def val_step(x_batch, y_prod, y_qty, y_price):
    # Model murni menebak kemampuannya saat ini tanpa bisa memperbaiki otaknya.
    # training=False berarti Dropout dimatikan, model memakai 100% konsentrasinya.
    pred_prod, pred_qty, pred_price = model(x_batch, training=False)

    # Tetap terapkan aturan masker yang sama agar evaluasinya adil
    mask_padding = tf.cast(tf.not_equal(x_batch, 0), tf.float32)
    mask_class = tf.where(tf.equal(y_prod, 0), 1.0, 10.0)
    final_mask = mask_padding * mask_class

    # Hitung kesalahan masing-masing cabang
    loss_prod_raw = loss_fn_product(y_prod, pred_prod)
    loss_prod = tf.reduce_sum(loss_prod_raw * final_mask) / (tf.reduce_sum(final_mask) + 1e-7)

    loss_qty = loss_fn_quantity(y_qty, pred_qty)
    loss_price = loss_fn_price(y_price, pred_price)

    # Di validation, kita hitung raw_total_loss murni tanpa dikalikan Dynamic Weighting.
    # agar sistem Early Stopping mendapat nilai asli
    raw_total_loss = loss_prod + loss_qty + loss_price

    val_acc_metric.update_state(y_prod, pred_prod, sample_weight=mask_padding)

    return raw_total_loss, loss_prod, loss_qty, loss_price

print("Fungsi train_step dan val_step berhasil dibuat")

Fungsi train_step dan val_step berhasil dibuat


## 7. Model Training

Pada tahap ini, kita akan menjalankan proses pelatihan selama beberapa Epoch. Di setiap epoch, model akan melakukan serangkaian aktivitas utama:

1. Siklus Pelatihan (Training): Model membaca seluruh data di train_ds, melakukan prediksi, menghitung kesalahan, dan memperbaiki dirinya sendiri menggunakan fungsi train_step yang sudah kita buat dengan tf.GradientTape.

2. Siklus Evaluasi (Validation): Setelah satu putaran belajar selesai, model diuji menggunakan data val_ds melalui fungsi val_step untuk melihat sejauh mana ia bisa menggeneralisasi pola tanpa melakukan perbaikan bobot.

3. Penyimpanan Otomatis & Early Stopping (Custom Callback): Karena kita menggunakan Custom Training Loop, kita menerapkan logika callback manual. Sistem akan selalu memantau Validation Loss. Jika performa membaik (memecahkan rekor), model akan otomatis disimpan. Namun, jika performa stagnan atau memburuk selama beberapa putaran berturut-turut (patience), sistem akan menarik rem darurat (Early Stopping) untuk menghemat waktu dan mencegah Overfitting.

Kita juga akan memantau nilai Akurasi untuk Produk serta nilai MAE (Mean Absolute Error) dari cabang Quantity dan Price. Ini penting untuk memastikan bahwa Dynamic Loss Weighting dan MaskedPriceLoss bekerja dengan efektif dalam menyeimbangkan prioritas belajar model.

Format Ekspor Model Final:
Model dengan nilai Validation Loss terbaik akan langsung diekspor secara senyap ke dalam dua format:

1. .keras: Format Keras V3 tunggal yang ideal untuk dokumentasi (AI-1) dan eksperimen/training lanjutan di Python.

2. SavedModel (Folder): Format universal production-ready yang siap digunakan oleh tim Inference. Format inilah yang akan diserahkan kepada AI-2 (Denny) untuk di-deploy ke Server API Backend.

In [8]:
# 1. KONFIGURASI PELATIHAN & INISIALISASI EARLY STOPPING
EPOCHS = 50  # Batas maksimal putaran pelatihan
history = [] # Buku catatan untuk menyimpan grafik progres model

# Variabel Early Stopping dan Model Checkpoint
# memantau Akurasi Produk
best_val_acc = 0.0      # Rekor nilai tertinggi dimulai dari 0%
patience = 7            # Batas toleransi model boleh gagal memecahkan rekor (7 putaran)
patience_counter = 0    # Penghitung jumlah kegagalan berturut-turut

# Siapkan folder untuk menyimpan model AI jika ia memecahkan rekor
os.makedirs("/content/drive/MyDrive/ChatKasir/assets/models", exist_ok=True)
save_path_keras = "/content/drive/MyDrive/ChatKasir/assets/models/chatkasir_model.keras"
save_path_best_sm = "/content/drive/MyDrive/ChatKasir/assets/models/chatkasir_saved_model"

print(f"Memulai Pelatihan selama {EPOCHS} Epoch...\n")

for epoch in range(EPOCHS):
    start_time = time.time() # Mulai stopwatch untuk menghitung durasi 1 epoch

    # 2. SIKLUS TRAINING
    # Model membaca seluruh data latih, menebak, dan memperbaiki otaknya.
    epoch_train_loss = epoch_l_prod = epoch_l_qty = epoch_l_price = 0.0
    num_train_batches = 0

    # Keluarkan data dari tf.data.Dataset per batch
    for x_batch, (y_prod, y_qty, y_price) in train_ds:
        # Jalankan 1 langkah training
        t_loss, l_prod, l_qty, l_price = train_step(x_batch, y_prod, y_qty, y_price)

        # Akumulasi semua kesalahan untuk dihitung rata-ratanya nanti
        epoch_train_loss += t_loss
        epoch_l_prod += l_prod
        epoch_l_qty += l_qty
        epoch_l_price += l_price
        num_train_batches += 1

    # Hitung rata-rata nilai Training
    avg_train_loss = epoch_train_loss / num_train_batches
    avg_train_prod = epoch_l_prod / num_train_batches
    avg_train_qty = epoch_l_qty / num_train_batches
    avg_train_price = epoch_l_price / num_train_batches

    # Ambil hasil akurasi training dari metrik Keras
    train_acc = train_acc_metric.result()

    # 3. SIKLUS VALIDATION
    # Model diuji dengan data baru yang belum pernah dilihat sebelumnya.
    epoch_val_loss = epoch_val_prod = epoch_val_qty = epoch_val_price = 0.0
    num_val_batches = 0

    for x_batch_val, (y_prod_val, y_qty_val, y_price_val) in val_ds:
        # Panggil fungsi val_step
        v_loss, v_l_prod, v_l_qty, v_l_price = val_step(x_batch_val, y_prod_val, y_qty_val, y_price_val)

        epoch_val_loss += v_loss
        epoch_val_prod += v_l_prod
        epoch_val_qty += v_l_qty
        epoch_val_price += v_l_price
        num_val_batches += 1

    # Hitung rata-rata nilai Validation
    avg_val_loss = epoch_val_loss / num_val_batches
    avg_val_prod = epoch_val_prod / num_val_batches
    avg_val_qty = epoch_val_qty / num_val_batches
    avg_val_price = epoch_val_price / num_val_batches

    # Ambil hasil akurasi validation
    val_acc = val_acc_metric.result()

    # 4. MONITORING DISPLAY
    duration = time.time() - start_time # Matikan stopwatch

    # Melihat Learning Rate saat ini dari dalam Optimizer
    current_lr = lr_schedule(optimizer.iterations).numpy()

    print(f"Epoch {epoch+1}/{EPOCHS} - {duration:.1f}s")
    print(f"Learning Rate: {current_lr:.6f}")

    print(f" > TOTAL Loss   : Train {avg_train_loss:.4f} | Val {avg_val_loss:.4f}")
    print(f"   > Loss Prod  : Train {avg_train_prod:.4f} | Val {avg_val_prod:.4f}")
    print(f"   > Loss Qty   : Train {avg_train_qty:.4f} | Val {avg_val_qty:.4f}")
    print(f"   > Loss Price : Train {avg_train_price:,.2f}   | Val {avg_val_price:,.2f}")
    print(f" > Prod Accuracy: Train {train_acc*100:.2f}% | Val {val_acc*100:.2f}%")
    print("-" * 50)

    # 5. EARLY STOPPING & MODEL CHECKPOINT
    # Jika Akurasi Validasi saat ini lebih tinggi dari rekor terbaik sebelumnya
    if val_acc > best_val_acc:
        print(f"Val Acc membaik dari {best_val_acc*100:.2f}% ke {val_acc*100:.2f}%!")
        print(f"Menyimpan model terbaik ke {save_path_best_sm}...")

        best_val_acc = val_acc # Perbarui rekor
        patience_counter = 0   # Reset jumlah kegagalan kembali ke 0

        # Ekspor model dalam 2 format (.keras & SavedModel)
        model.save(save_path_keras)

        with open(os.devnull, 'w') as f, contextlib.redirect_stdout(f):
            model.export(save_path_best_sm)

    else:
        # Jika rekor tidak terpecahkan, tambah peringatan.
        patience_counter += 1
        print(f"Val Acc tidak membaik. Kesempatan: {patience_counter}/{patience}")

    # 6. DYNAMIC LOSS WEIGHTING (3 Fase)
    # Mengontrol tugas mana yang harus dihukum lebih keras di putaran selanjutnya
    is_prod_passed = val_acc >= 0.95
    is_qty_passed = avg_val_qty <= 0.02

    current_w_prod = w_product.numpy()
    current_w_qty = w_quantity.numpy()
    current_w_price = w_price.numpy()

    if not is_prod_passed:
        # FASE 1: Fokus selesaikan Akurasi Produk
        print("    > FASE 1 AKTIF: Fokus Penuh ke Product")
        w_quantity.assign(0.1) # Abaikan kesalahan kuantitas
        w_price.assign(0.1)    # Abaikan kesalahan harga

        new_w_prod = min(current_w_prod + 1.0, 15.0)
        w_product.assign(new_w_prod)
        print(f"    > Product: GAGAL (<95%), penalti ditambahkan. Bobot = {new_w_prod:.1f}")

    elif not is_qty_passed:
        # --- FASE 2: Produk lulus, Pindah fokus ke Kuantitas ---
        print("    > FASE 2 AKTIF: Product Aman, Fokus Penuh ke Quantity")
        w_product.assign(1.0)  # Istirahatkan beban produk (kembali ke normal)
        w_price.assign(0.1)

        new_w_qty = min(current_w_qty + 1.0, 15.0)
        w_quantity.assign(new_w_qty)
        print(f"    > Qty   : GAGAL (MAE > 0.02), penalti ditambahkan. Bobot = {new_w_qty:.1f}")

    else:
        # --- FASE 3: Produk & Kuantitas lulus, Fokus ke Harga ---
        print("    > FASE 3 AKTIF: Product & Quantity Aman, Fokus Penuh ke Price")
        w_product.assign(1.0)
        w_quantity.assign(1.0)

        if avg_val_price <= 0.02:
            w_price.assign(0.5)
            print("    > STATUS: SEMUA TARGET AKURASI & LOSS TERCAPAI")
        else:
            new_w_price = min(current_w_price + 1.0, 15.0)
            w_price.assign(new_w_price)
            print(f"    > Price : GAGAL (MAE >0.02), penalti ditambahkan. Bobot = {new_w_price:.1f}")

    print("-" * 50)

    # 7. CLEANUP & PENGECEKAN EARLY STOPPING AKHIR
    # Kosongkan metric akurasi agar siap dipakai di epoch berikutnya
    train_acc_metric.reset_state()
    val_acc_metric.reset_state()

    # Catat semua nilai ke history agar nanti bisa ditampilkan grafiknya
    history.append({
        'train_loss': avg_train_loss.numpy(), 'val_loss': avg_val_loss.numpy(),
        'train_acc': train_acc.numpy(), 'val_acc': val_acc.numpy()
    })

    # Early Stopping jika jatah kesempatannya sudah habis
    if patience_counter >= patience:
        print(f"\nPelatihan dihentikan otomatis pada Epoch {epoch+1}.")
        print("Model sudah tidak membaik selama 7 putaran berturut-turut.")
        break

print("\nProses Pelatihan Selesai!")

Memulai Pelatihan selama 50 Epoch...

Epoch 1/50 - 68.4s
Learning Rate: 0.001000
 > TOTAL Loss   : Train 5.0922 | Val 1.3344
   > Loss Prod  : Train 0.0764 | Val 0.0148
   > Loss Qty   : Train 0.5278 | Val 0.2672
   > Loss Price : Train 4.49   | Val 1.05
 > Prod Accuracy: Train 97.11% | Val 99.81%
--------------------------------------------------
Val Acc membaik dari 0.00% ke 99.81%!
Menyimpan model terbaik ke /content/drive/MyDrive/ChatKasir/assets/models/chatkasir_saved_model...
    > FASE 2 AKTIF: Product Aman, Fokus Penuh ke Quantity
    > Qty   : GAGAL (MAE > 0.02), penalti ditambahkan. Bobot = 2.0
--------------------------------------------------
Epoch 2/50 - 50.9s
Learning Rate: 0.000900
 > TOTAL Loss   : Train 0.4367 | Val 1.5854
   > Loss Prod  : Train 0.0118 | Val 0.0114
   > Loss Qty   : Train 0.1686 | Val 0.1787
   > Loss Price : Train 0.88   | Val 1.40
 > Prod Accuracy: Train 99.83% | Val 99.82%
--------------------------------------------------
Val Acc membaik dari 99.8